# Pipeline Project

You will be using the provided data to create a machine learning model pipeline.

You must handle the data appropriately in your pipeline to predict whether an
item is recommended by a customer based on their review.
Note the data includes numerical, categorical, and text data.

You should ensure you properly train and evaluate your model.

## The Data

The dataset has been anonymized and cleaned of missing values.

There are 8 features for to use to predict whether a customer recommends or does
not recommend a product.
The `Recommended IND` column gives whether a customer recommends the product
where `1` is recommended and a `0` is not recommended.
This is your model's target/

The features can be summarized as the following:

- **Clothing ID**: Integer Categorical variable that refers to the specific piece being reviewed.
- **Age**: Positive Integer variable of the reviewers age.
- **Title**: String variable for the title of the review.
- **Review Text**: String variable for the review body.
- **Positive Feedback Count**: Positive Integer documenting the number of other customers who found this review positive.
- **Division Name**: Categorical name of the product high level division.
- **Department Name**: Categorical name of the product department name.
- **Class Name**: Categorical name of the product class name.

The target:
- **Recommended IND**: Binary variable stating where the customer recommends the product where 1 is recommended, 0 is not recommended.

## Load Data

In [ ]:
import pandas as pd
from pandas.core.common import random_state

# Load data
df = pd.read_csv(
    'data/reviews.csv',
)

df.info()
df.head()

## Preparing features (`X`) & target (`y`)

In [ ]:
data = df

# separate features from labels
X = data.drop('Recommended IND', axis=1)
y = data['Recommended IND'].copy()

print('Labels:', y.unique())
print('Features:')
display(X.head())

In [ ]:
# Split data into train and test sets
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.1,
    shuffle=True,
    random_state=27,
)

# Your Work

## Data Exploration

In [ ]:
X_train.describe()

In [ ]:
X_train.shape
X_train.head()

In [ ]:
import missingno as msno

msno.matrix(X_train)
msno.bar(X_train)
msno.heatmap(X_train)
msno.dendrogram(X_train)

In [ ]:
y_train.value_counts()
y_train.value_counts(normalize=True)

In [ ]:
import seaborn as sns

sns.set_theme(style="darkgrid")
sns.histplot(X_train['Age'])
sns.displot(X_train['Positive Feedback Count'])
#sns.catplot(data=X_train, kind="box")

In [ ]:
sns.catplot(data=X_train, kind="box")

In [ ]:
X_train.columns

display(X_train['Division Name'].unique())
display(X_train['Department Name'].unique())
display(X_train['Class Name'].unique())

In [ ]:
#X_train[X_train['Class Name'] == 'Trend'].shape

print(X_train['Department Name'].value_counts())
print(X_train['Class Name'].value_counts())

In [ ]:
print(X_train.duplicated().sum())

In [ ]:
X_train.columns

## Building Pipeline

In [ ]:
from sklearn.pipeline import Pipeline
import numpy as np

num_features = (
    X_train
    .select_dtypes(exclude=['object']).columns
    .drop(
        [
            'Clothing ID', # More of category than a numerical feature
            'Positive Feedback Count'
        ],
    )
)
print('Numerical features:', num_features)


num_features_log = (
    X_train[[
        'Positive Feedback Count',
        ]].columns
)

print('Log Numerical features:', num_features_log)

cat_features = (
    X_train[[
        'Division Name',
        'Department Name',
        'Class Name',
    ]].columns
)
print('Categorical features:', cat_features)


text_features = (
    X[[
        'Title',
        'Review Text',
    ]].columns
)
print ('Review Text features:', text_features)

## Training Pipeline

In [ ]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin
import spacy, click

nlp = spacy.load('en_core_web_sm')


num_pipeline = Pipeline([
    (
        'scaler',
        StandardScaler(),
    ),
])

log_transformer =  FunctionTransformer(np.log1p, validate=False)

num_pipeline_log = Pipeline([
    (
        'log1p',
        log_transformer,
    ),
    (
        'scaler',
        StandardScaler(),
    ),
])

cat_pipeline = Pipeline([
    (
        'cat_encoder',
        OneHotEncoder(
            sparse_output=False,
            handle_unknown='ignore',)
    ),
])

class Merger(BaseEstimator, TransformerMixin):
    def __init__(self, Col1, Col2):
        self.Col1 = Col1
        self.Col2 = Col2

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        merged = X[self.Col1] + ' ' +X[self.Col2]
        return merged

class SpacyLemmatizer(BaseEstimator, TransformerMixin):
    def __init__(self, nlp):
        self.nlp = nlp

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        lemmatized = [
            ' '.join(
                token.lemma_ for token in doc
                if not token.is_stop
            )
            for doc in self.nlp.pipe(X)
        ]
        return lemmatized


tfidf_pipeline = Pipeline([
    (
       'merger',
        Merger(Col1='Title', Col2='Review Text'),
    ),
    (
       'lemmatizer',
        SpacyLemmatizer(nlp=nlp),
    ),
    (
        'tfidf_vectorizer',
        TfidfVectorizer(
            stop_words='english',
        ),
    ),
])

feature_engineering = ColumnTransformer([
        ('num', num_pipeline, num_features),
        ('num_log', num_pipeline_log, num_features_log),
        ('cat', cat_pipeline, cat_features),
        ('tfidf_text', tfidf_pipeline, text_features),
])

feature_engineering

In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

model_pipeline = make_pipeline(
    feature_engineering,
    LogisticRegression(random_state=27, solver='saga'),
)

clf = model_pipeline.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import roc_auc_score
roc_auc_score(y_test, clf.predict_proba(X_test)[:, 1])

## Fine-Tuning Pipeline

In [ ]:
pd.DataFrame(param_search.cv_results_)

In [ ]:
from sklearn.model_selection import GridSearchCV
clf.named_steps

param_grid = {
    'columntransformer__tfidf_text__tfidf_vectorizer__ngram_range': [(1,2), (2,1), (2,2)],
    'logisticregression__penalty': ['l1', 'l2']
}

#'columntransformer__tfidf_text__tfidf_vectorizer__max_df': [0.7, 0.9],
#'columntransformer__tfidf_text__tfidf_vectorizer__min_df': [1, 3],


param_search = GridSearchCV(
    estimator=clf,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=3,
    n_jobs=-1,
    refit=True,
    return_train_score=True,
    verbose=True,
)


param_search.fit(X_train, y_train)

# Retrieve the best parameters
param_search.best_params_